*0.2 Math / ML basics*

# overfitting

**The situation.** A classifier sorts forum posts into three topics. On the 30 posts used to train it: 100% correct. On the next week's posts: about half. The gap between those two numbers is the diagnosis. The model learned the 30 posts — their exact words, their quirks — instead of the pattern that carries over. That is *overfitting*.

**Seeing it.** Train the same model on 30, 100, 300, 1,000 and all available posts. Training accuracy stays near 100% whatever the size — a model can always memorise a small set. Validation accuracy tells the truth, and the gap between the two is the amount of memorising. Real data here: the classic 20 Newsgroups posts, three topics, headers and quotes stripped.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

topics = ["sci.med", "sci.space", "rec.autos"]
train = fetch_20newsgroups(
    subset="train", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
validation = fetch_20newsgroups(
    subset="test", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
print("posts available:", len(train.data), "train,", len(validation.data), "validation")

print(f"{'train size':>11}{'train acc':>12}{'val acc':>10}{'gap':>8}")
gaps = {}
for size in (30, 100, 300, 1000, len(train.data)):
    model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=10, max_iter=3000))
    model.fit(train.data[:size], train.target[:size])
    train_accuracy = model.score(train.data[:size], train.target[:size])
    val_accuracy = model.score(validation.data, validation.target)
    gaps[size] = train_accuracy - val_accuracy
    print(f"{size:>11}{train_accuracy:>12.0%}{val_accuracy:>10.0%}{gaps[size]:>8.0%}")
assert gaps[30] > 0.3 and gaps[len(train.data)] < gaps[30]

posts available: 1781 train, 1186 validation
 train size   train acc   val acc     gap
         30        100%       51%     49%
        100        100%       58%     42%


        300         99%       80%     19%


       1000         98%       85%     13%


       1781         98%       87%     11%


**Reading the output.** Training accuracy is near 100% at every size — it says nothing. Validation accuracy climbs as the model sees more posts, and the gap shrinks from around half to around a tenth. The 30-post model was not good; it had memorised 30 posts.

```
accuracy
 100% │ ●────●────●────●────●   train (always high — meaningless on its own)
      │                    ○
      │              ○
      │        ○                 validation (the truth)
  50% │  ○                       gap = overfitting
      └────────────────────────── more training data →
```

**The rule to remember.** Judge a model by validation accuracy, never training accuracy. A large gap means memorising; the fixes are more data, or a penalty on complexity (next item).

| Use it when | Don't when | Instead use |
|---|---|---|
| — this is a diagnosis, run it on every model you train | — | — |

**Watch out**
- Fine-tuning an LLM on 200 examples for 10 epochs overfits exactly like this; watch validation loss per epoch and stop when it stops falling.
- Few-shot prompts overfit too: examples chosen to pass your 20 test cases can hurt on the next 2,000.
- Underfitting is the opposite (both accuracies low) — the fix is more freedom, not less.